In [ ]:
#kc-event-explorer/
#│
#├── WIP- 1.A.py                  
#├── templates/
#│   └── index.html
#│
#├── static/
#│   ├── style.css
#│   └── script.js
#│
#├── data/
#│   └── neighborhoods.geojson
#│
#├── services/
#│   ├── ticketmaster.py
#│   └── geocoder.py
#│
#└── database/
#    └── events.db

In [4]:
import requests
from flask import Flask, render_template, request
from keys import TICKETMASTER_KEY, EVENTBRITE_TOKEN

app = Flask(__name__)


def get_ticketmaster_events():
    url = "https://app.ticketmaster.com/discovery/v2/events.json"

    params = {
       "apikey": TICKETMASTER_KEY,
         "city": "Kansas City",
           "countryCode": "US",
           "size": 50
    } 
    
    response = requests.get(url, params=params)
    return response.json()

def get_eventbrite_events():
    url = "https://www.eventbriteapi.com/v3/events/search/"

    headers = {
        "Authorization": f"Bearer {EVENTBRITE_TOKEN}"
    }

    params = {
        "location.address": "Kansas City",
        "page_size": 50
    }

    response = requests.get(url, headers=headers, params=params)
    return response.json()


def normalize_ticketmaster(data):
    events = []

    if "_embedded" in data:
        for event in data["_embedded"]["events"]:

            start = event.get("dates", {}).get("start", {})

            raw_time = start.get("localTime")
            time = raw_time[:5] if raw_time else None

            venue = {}

            if "_embedded" in event:
                venues = event["_embedded"].get("venues", [])
                if venues:
                    venue = venues[0]

            events.append({
                "name": event.get("name"),
                "date": start.get("localDate"),
                "time": time,
                "source": "ticketmaster",
                "lat": venue.get("location", {}).get("latitude"),
                "lon": venue.get("location", {}).get("longitude")
            })

    return events


def normalize_eventbrite(data):
    events = []

    for event in data.get("events", []):

        name = event.get("name", {}).get("text")
        start = event.get("start", {}).get("local")

        date = None
        time = None

        if start:
            date = start[:10]
            time = start[11:16]

        events.append({
            "name": name,
            "date": date,
            "time": time,
            "source": "eventbrite",
            "lat": None,
            "lon": None
        })

    return events

from datetime import datetime

def get_all_events():
    tm_data = get_ticketmaster_events()
    eb_data = get_eventbrite_events()

    events = []

    events += normalize_ticketmaster(tm_data)
    events += normalize_eventbrite(eb_data)

    return events


@app.route("/")
def home():

    print("ROUTE HIT")

    selected_date = request.args.get("date")
    selected_time = request.args.get("time")

    data = get_all_events()  

    events = []

    for event in data:

        name = event.get("name")
        date = event.get("date")
        time = event.get("time")

        if selected_date and date != selected_date:
            continue

        if selected_time and time and time < selected_time:
            continue

        display_time = "TBD"

        if time:
            try:
             display_time = datetime.strptime(
                time,
                "%H:%M"
                ).strftime("%I:%M %p")
            except:
                display_time = time

        events.append({
            "name": name,
            "date": date,
         "time": display_time,
            "source": event.get("source"),
            "lat": event.get("lat"),
            "lon": event.get("lon")
        })

    print("EVENT COUNT:", len(events))

    return render_template(
        "index.html",
        events=events,
        selected_date=selected_date,
        selected_time=selected_time
    )


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5001, debug=True, use_reloader=False)




 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


ROUTE HIT


127.0.0.1 - - [22/Jun/2026 07:41:15] "GET /?date=2026-06-25&time= HTTP/1.1" 200 -


EVENT COUNT: 1
